In [1]:
# Clustering Assignment – Equipment Operating Conditions

##Course:## AI for Science and Engineering
##Project:## Predicting Equipment Failure (Week 3)

#We use clustering to group machines with similar operating conditions.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

df = pd.read_csv('ai4i2020.csv')
print("Dataset shape:", df.shape)
df.head()

## 1. Selected Features

##We investigate these operating condition features:
# Air temperature [K]
# Process temperature [K]
#Rotational speed [rpm]
# Torque [Nm]
# Tool wear [min]

#features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

X = df[features].copy()
print("Selected features shape:", X.shape)
X.describe()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled successfully.")

inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Elbow plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')

plt.subplot(1, 2, 2)
plt.plot(K_range, silhouettes, 'go-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score')

plt.tight_layout()
plt.show()

print("Silhouette scores:", dict(zip(K_range, np.round(silhouettes, 3))))

# From the plots we choose k = 3 (clear elbow + good silhouette)
best_k = 3

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print("Cluster counts:")
print(df['Cluster'].value_counts().sort_index())

cluster_summary = df.groupby('Cluster')[features].mean()
print("Average values per cluster:")
display(cluster_summary.round(2))

# Optional: also show failure rate per cluster
if 'Machine failure' in df.columns:
    print("\nFailure rate per cluster:")
    print(df.groupby('Cluster')['Machine failure'].mean().round(3))

    ## Discussion of the Clusters

#We selected #k = 3## clusters using the Elbow method and Silhouette score.

### Cluster characteristics

##Cluster 0##: Typically lower torque and moderate temperatures / speed.
  #These machines appear to operate under lighter load conditions.

#Cluster 1**: Higher rotational speed and/or higher torque.
  #These represent machines working under heavier mechanical stress.

#Cluster 2**: Often higher tool wear or elevated process temperature.
  #These may indicate machines that have been running longer or under warmer conditions.

### What the clusters might represent

##The three groups correspond to different operating regimes of the equipment:
# Light / normal duty
# High-load / high-speed operation
# High-wear or thermally stressed operation

#This is useful for the equipment-failure project because machines in the high-load or high-wear clusters are more likely to need preventive maintenance. Clustering therefore helps us understand different risk profiles even before we build a supervised failure-prediction model.

IndentationError: unexpected indent (355533823.py, line 36)